# Exploring the Knowledge Graph

### Definition of functions to access, query and plot the knowledge graph/subgraphs

In [ ]:
from py2neo import Graph
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd

def getGraphDatabaseConnectionConfiguration():
    host = config.dbURL
    port = config.dbPort
    user = config.dbUser
    password = config.dbPassword

    driver = connectToDB(host, port, user, password)

    return driver

def connectToDB(host = "localhost", port= 7687, user="neo4j", password = "password"):
    driver = Graph(host=host, port = port, user = user, password = password)

    return driver

def simplePlotGraph(graph):
    G=nx.Graph()
    for rel in graph["relationships"]:
        snode = rel.start_node()
        enode = rel.end_node()
        if "name" in snode and "name" in enode:
            G.add_edge(snode["name"],enode["name"], label=rel.type())
    plt.figure(3,figsize=(25,25)) 
    pos=nx.spring_layout(G)
    nx.draw_networkx(G, pos,node_size=260,font_size=16)
    plt.axis('off')
    plt.show()
    
def getCursorData(cursor, header):
    df = pd.DataFrame(cursor)
    return df

### Definition of Cypher queries to extract information from the graph

In [ ]:
dbstats_queries = {"num_nodes":'''MATCH (n) RETURN count(n);''',
           "num_relationships":'''MATCH ()-->() RETURN count(*);''',
           "type_nodes":'''MATCH (n) WHERE rand() <= 0.1
                           RETURN
                           DISTINCT labels(n),
                           count(*) AS SampleSize,avg(size(keys(n))) as Avg_PropertyCount, 
                           min(size(keys(n))) as Min_PropertyCount,
                           max(size(keys(n))) as Max_PropertyCount,
                           avg(size( (n)-[]-() ) ) as Avg_RelationshipCount,
                           min(size( (n)-[]-() ) ) as Min_RelationshipCount,
                           max(size( (n)-[]-() ) ) as Max_RelationshipCount;''',
           "schema":'''CALL db.schema();'''}
project_queries = {"existing_projects": '''MATCH (n:Project) RETURN n.name AS Name, n.acronym AS acronym, n.responsible AS Responsible''',
                  "subjects": '''MATCH (p:Project)-[r:HAS_ENROLLED]->(s:Subject) RETURN p.name AS Project, s.id AS Id, s.external_id AS External'''}
analytical_samples = {"proteins": '''MATCH (a:Analytical_sample{id:"IDENTIFIER"})-[r:HAS_QUANTIFIED_PROTEIN]->(p:Protein) 
                                    RETURN a.id AS Analytical_sampleId, p.id AS ProteinId, p.name AS Name, 
                                    p.synonyms AS Synonyms, r.value AS Intensity LIMIT 25''',
                     "peptides": '''MATCH (a:Analytical_sample{id:"IDENTIFIER"})-[r:HAS_QUANTIFIED_PEPTIDE]->(p:Peptide) 
                                    RETURN a.id AS Analytical_sampleId, p.id AS PeptideId, p.name AS Name, 
                                    r.value AS Intensity LIMIT 25''',
                     "ptm": '''MATCH (a:Analytical_sample{id:"IDENTIFIER"})-[r:HAS_QUANTIFIED_PROTEINMODIFICATION]->(p:Modified_protein) 
                             RETURN a.id AS Analytical_sampleId, p.id AS PTMId, 
                             p.protein AS Protein, p.position AS position , p.residue as Residue LIMIT 25'''}
mentions_queries = {"disease":'''MATCH (d:Disease{id:"DISEASE"})-[r:MENTIONED_IN_PUBLICATION]->(pubmed:Publication) RETURN d.id AS Id, d.name AS Name, pubmed.id AS PUBMEDID LIMIT 25'''}
variant_queries = {"subject":'''MATCH (a:Analytical_sample)-[r:CALLED_VARIANT]->(s:Somatic_mutation) RETURN a.id AS AnalyticalSampleId, s.id AS SomaticMutationId  LIMIT 25'''}

## Connect to the graph

In [ ]:
graph = connectToDB( user="neo4j", password = "bioinfo1112")

### Access some of the graph statistics

In [ ]:
getCursorData(cursor = graph.run(dbstats_queries["num_nodes"]).data(), header = ["Number of Nodes"])

In [ ]:
getCursorData(cursor = graph.run(dbstats_queries["num_relationships"]).data(), header = ["Number of Relationships"])

In [ ]:
getCursorData(cursor = graph.run(dbstats_queries["type_nodes"]).data(), header = ["Types"])

### Plot the data model

In [ ]:
subgraph = graph.run(dbstats_queries["schema"]).data()[0]
simplePlotGraph(subgraph)

### Projects already in the database

In [ ]:
getCursorData(cursor = graph.run(project_queries["existing_projects"]).data(), header = ["Types"])

In [ ]:
getCursorData(cursor = graph.run(project_queries["subjects"]).data(), header = ["Types"]).set_index(["Project", "Id"]).sort_values(by="External")

In [ ]:
getCursorData(cursor = graph.run(analytical_samples["proteins"].replace("IDENTIFIER", "AS22")).data(), header = ["Types"]).set_index(["Analytical_sampleId", "ProteinId"]).sort_values(by="Name")

In [ ]:
getCursorData(cursor = graph.run(analytical_samples["peptides"].replace("IDENTIFIER", "AS22")).data(), header = ["Types"]).set_index(["Analytical_sampleId", "PeptideId"]).sort_values(by="Name")

In [ ]:
getCursorData(cursor = graph.run(analytical_samples["ptm"].replace("IDENTIFIER", "AS22")).data(), header = ["Types"]).set_index(["Analytical_sampleId", "PTMId"]).sort_values(by="Protein")

In [ ]:
df = getCursorData(cursor = graph.run(mentions_queries["disease"].replace("DISEASE", "DOID:0014667")).data(), header = ["Types"]).set_index(["Id", "Name"]).sort_values(by="PUBMEDID")
df["linkout"] = "https://www.ncbi.nlm.nih.gov/pubmed/"+df["PUBMEDID"]
df

In [ ]:
getCursorData(cursor = graph.run(variant_queries["subject"]).data(), header = ["Types"]).set_index(["AnalyticalSampleId", "SomaticMutationId"])